<a href="https://colab.research.google.com/github/eduardokern/ML/blob/face_detection/notebooks/face_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Criando um sistema de reconhecimento facial do zero</h1>

O objetivo principal deste projeto é trabalhar com as bibliotecas e frameworks estudados e analisados em nossas aulas. Neste sentido, a proposta padrão envolve um sistema de detecção e reconhecimento de faces, utilizando o framework TensorFlow em conjuntos com as bibliotecas que o projetista julgue necessárias, de forma ilimitada.  

Por meio da Figura 1 é possível visualizar o resultado esperado para o modelo proposto, devendo detectar e reconhecer mais de uma face ao mesmo tempo.  
Para isso você deve:

1. Utilizar uma rede de detecção treinada para detectar faces.
2. Utilizar uma rede de classificação para classificar a face detectada.

![Figura 1: Detecção e reconhecimento facial.](https://drive.google.com/uc?export=view&id=1nFCZd-FR0jIYAvDbDbVpDt6ansjEkLIA)

Para realizar este projeto, você pode utilizar os seguintes trabalhos de referência:
Detecção Facial:
https://colab.research.google.com/drive/1QnC7lV7oVFk5OZCm75fqbLAfD9qBy9bw?usp=sharing

Detecção e classificação de objetos:  
https://colab.research.google.com/drive/1xdjyBiY75MAVRSjgmiqI7pbRLn58VrbE?usp=sharing

<h3>Mount Google Drive to get datasets</h3>

In [11]:
# Mount google drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

datasets_path = '/content/drive/MyDrive/Colab/ML/datasets'

Mounted at /content/drive


<h3>Required imports</h3>

In [37]:
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import keras
from keras.applications.imagenet_utils import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
from keras.models import Sequential
from keras.preprocessing import image

<h3>Helper functions</h3>

In [3]:
# helper function to load image and return it and input vector
def get_image(path):
  try:
    img = image.load_img(path, target_size=(224, 224), keep_aspect_ratio=True)
    data = image.img_to_array(img)
    data = np.expand_dims(data, axis=0)
    data = preprocess_input(data)
    return img, data
  except Exception as e:
    print(f'Error loading: {path}, {e}')
    return None, None

In [15]:
# helper function to load detection dataset, image data and bbox
def load_dataset(path):
    dataset = []
    with open(f'{path}/_annotations.csv') as file:
        file.readline()
        for line in file.readlines():
            values = line.replace('\n', '').split(',')
            if len(values) > 1:
                filename = values[0]
                width = int(values[1])
                height = int(values[2])
                label = values[3]
                xmin = int(values[4])
                ymin = int(values[5])
                xmax = int(values[6])
                ymax = int(values[7])

                image_path = f'{path}/{filename}'
                image, data = get_image(image_path)
                dataset.append({'x':np.array(data[0]), 'y':[xmin, ymin, xmax, ymax], 'image': image})

    return dataset

<h3>FaceDetection class</h3>

It contais a CNN to detect all faces and their bbox and a VGG16 neural network to classify the faces detected.

Both NN can be trained with different datasets.

In [44]:
class FaceDetection:
  def __init__(self, detection_train_data, detection_val_data):
    self._detection_model(detection_train_data[0].shape[1:])
    self._train_detection(detection_train_data, detection_val_data, 10, 128)

  def _detection_model(self, input_shape):
    print(input_shape)

    self._detection_model = Sequential()
    self._detection_model.add(Input(shape=input_shape))
    self._detection_model.add(Conv2D(32, (3, 3), activation='relu'))
    self._detection_model.add(Conv2D(64, (3, 3), activation='relu'))
    self._detection_model.add(Flatten())
    self._detection_model.add(Dense(128, activation='relu'))
    # Output layer for bounding box coordinates
    self._detection_model.add(Dense(4))


  def _train_detection(self, train_data, val_data, epochs, batch_size):
    self._detection_model.compile(optimizer='adam', loss='mean_square_error', metrics=['accuracy'])
    self._detection_model.fit(
        train_data[0],
        train_data[1],
        validation_data=val_data,
        epochs=epochs,
        batch_size=batch_size)


  def detect(self, image):
    return self._detection_model.predict(np.array([image]))


  def train_classification(self, num_classes, train_data, val_data, epochs, batch_size):
    vgg = keras.applications.VGG16(weights='imagenet', include_top=True)
    inp = vgg.input

    # make a new softmax layer with num_classes neurons
    new_classification_layer = Dense(num_classes, activation='softmax')

    # connect our new layer to the second to last layer in VGG, and make a reference to it
    out = new_classification_layer(vgg.layers[-2].output)

    # create a new network between inp and out
    self._classification_model = Model(inp, out)

    # make all layers untrainable by freezing weights (except for last layer)
    for l, layer in enumerate(self._classification_model.layers[:-1]):
        layer.trainable = False

    # ensure the last layer is trainable/not frozen
    for l, layer in enumerate(self._classification_model.layers[-1:]):
        layer.trainable = True

    self._classification_model.compile(
        loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    self._classification_model.fit(
        train_data[0],
        train_data[1],
        validation_data=val_data,
        epochs=epochs,
        batch_size=batch_size)


  def predict(self, image):
    faces = self._detection_model.predict(np.array([image]))


Load face detection dataset to train the face detection CNN to identify all faces and their bbox in an image.

In [21]:
detection_dataset_path = f'{datasets_path}/FaceDetection'
detection_train = load_dataset(f'{detection_dataset_path}/train')
detection_valid = load_dataset(f'{detection_dataset_path}/valid')
detection_test = load_dataset(f'{detection_dataset_path}/test')

x_detection_train, y_detection_train = np.array([t["x"] for t in detection_train]), [t["y"] for t in detection_train]
x_detection_valid, y_detection_valid = np.array([t["x"] for t in detection_valid]), [t["y"] for t in detection_valid]

x_detection_train = x_detection_train.astype('float32') /255.
x_detection_valid = x_detection_train.astype('float32') /255.

In [43]:
detector = FaceDetection(
    (x_detection_train, y_detection_train),
    (x_detection_valid, y_detection_valid))

image, data = get_image(f'{datasets_path}/big3/big3/big3-1.jfif')
detector.detect(data)

(224, 224, 3)


ValueError: Unrecognized data type: x=[[[[-0.2625059  -0.29325098 -0.2850196 ]
   [-0.2625059  -0.29325098 -0.2850196 ]
   [-0.25466275 -0.28540784 -0.27717647]
   ...
   [ 0.30612156  0.29106274  0.2992941 ]
   [ 0.29827842  0.2832196   0.29145098]
   [ 0.29435685  0.27929804  0.2875294 ]]

  [[-0.2625059  -0.29325098 -0.2850196 ]
   [-0.2625059  -0.29325098 -0.2850196 ]
   [-0.2625059  -0.29325098 -0.2850196 ]
   ...
   [ 0.30612156  0.29106274  0.2992941 ]
   [ 0.29827842  0.2832196   0.29145098]
   [ 0.29435685  0.27929804  0.2875294 ]]

  [[-0.2625059  -0.29325098 -0.2850196 ]
   [-0.2625059  -0.29325098 -0.2850196 ]
   [-0.25858432 -0.2893294  -0.28109804]
   ...
   [ 0.30612156  0.29106274  0.2992941 ]
   [ 0.29827842  0.2832196   0.29145098]
   [ 0.29435685  0.27929804  0.2875294 ]]

  ...

  [[ 0.40808234  0.34988627  0.32282352]
   [ 0.40808234  0.34988627  0.32282352]
   [ 0.40416077  0.3459647   0.31890196]
   ...
   [ 0.0512196   0.06361177  0.064     ]
   [ 0.0512196   0.